# PARSeq training on Colab — Drive patch version

This notebook is set up for your current PARSeq fine-tuning workflow, but with Colab/Drive paths and GPU training.

It does five main things:

1. Mounts Google Drive.
2. Clones the stock `baudm/parseq` repo.
3. Installs the PARSeq training dependencies.
4. Applies your `parseq_hybrid_tokenizer_patch` zip/folder from Google Drive into the cloned repo.
5. Trains with `charset=latex_hybrid` by default.

**Expected Drive dataset layout**

Put your LMDB data in:

```text
/content/drive/MyDrive/Parseq/Training/data/
├── train/
│   └── real/
│       └── your_train_dataset/
│           ├── data.mdb
│           └── lock.mdb
└── val/
    └── real/
        └── your_val_dataset/
            ├── data.mdb
            └── lock.mdb
```

The dataset subfolder names are flexible. PARSeq recursively searches for `data.mdb`.

**Expected Drive patch location**

The notebook will first use `PATCH_SOURCE_STR` from the config cell. If that path does not exist, it searches common locations such as:

```text
/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch_20260524.zip
/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch.zip
/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch/
```

**Runtime setting**

In Colab, use:

```text
Runtime → Change runtime type → T4 GPU
```


In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Configure paths and training options

from pathlib import Path

# Edit this if you want a different Drive folder.
DRIVE_TRAINING_DIR_STR = "/content/drive/MyDrive/Parseq/Training"  #@param {type:"string"}
DRIVE_TRAINING_DIR = Path(DRIVE_TRAINING_DIR_STR)

# Your Drive dataset folder. Expected to contain train/ and val/.
DATA_SOURCE_DIR = DRIVE_TRAINING_DIR / "data"

# Local copies are much faster than training directly from Drive.
WORK_DATA_DIR = Path("/content/parseq_data")

# Where checkpoints/TensorBoard logs should be saved.
RUNS_DIR = DRIVE_TRAINING_DIR / "runs"

# PARSeq repo location in the Colab VM.
PARSEQ_DIR = Path("/content/parseq")

# Hybrid-tokenizer patch on Drive.
# This can be either the zip file OR the extracted folder shown in Finder.
PATCH_SOURCE_STR = "/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch_20260524.zip"  #@param {type:"string"}
PATCH_SOURCE = Path(PATCH_SOURCE_STR)

# Current training settings.
# `latex_hybrid` is created by the Drive patch.
CHARSET = "latex_hybrid"         # hybrid LaTeX charset from the patch
MAX_LABEL_LENGTH = 44             # measured in hybrid tokens when CHARSET=latex_hybrid
BATCH_SIZE = 32
MAX_EPOCHS = 200
NUM_WORKERS = 2
LOG_EVERY_N_STEPS = 5

# Safe to leave in the command. Patched train.py versions that support it will use it;
# patched train.py versions that do not support it will simply ignore the extra config key.
PRETRAINED_LOAD = "auto"

# Filled in after dependencies are installed and torch is imported.
ACCELERATOR = "auto"
DEVICES = 1

print("Drive training dir:", DRIVE_TRAINING_DIR)
print("Data source dir:   ", DATA_SOURCE_DIR)
print("Local data dir:    ", WORK_DATA_DIR)
print("Runs dir:          ", RUNS_DIR)
print("PARSeq dir:        ", PARSEQ_DIR)
print("Patch source:      ", PATCH_SOURCE)
print("Charset:           ", CHARSET)
print("Max label length:  ", MAX_LABEL_LENGTH)


In [ ]:
# Check whether Colab assigned a GPU.
# This does not import torch yet, so dependency installs can happen before Python loads NumPy/Torch.
!nvidia-smi || true

## Install PARSeq and dependencies

This uses the dependency pins from your local setup where they matter most for your workflow:

- `numpy==1.26.4`
- `opencv-python-headless==4.10.0.84`
- `imgaug==0.4.0`

Colab already comes with a CUDA PyTorch build, so this notebook **does not reinstall torch/torchvision** by default.

In [ ]:
%%bash
set -euo pipefail

python -m pip install --upgrade -q pip setuptools wheel

echo "Installing pinned NumPy/OpenCV stack..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84"

echo "Installing PARSeq training dependencies..."
python -m pip install -q \
  "pytorch-lightning>=2.2,<3" \
  "hydra-core>=1.3,<1.4" \
  "omegaconf>=2.3,<2.4" \
  "torchmetrics>=1.3,<2" \
  "tensorboard" \
  "timm" \
  "einops" \
  "nltk" \
  "lmdb" \
  "fire" \
  "pillow" \
  "matplotlib" \
  "pandas" \
  "tqdm" \
  "rich"

echo "Installing imgaug with compatible pinned dependencies..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84" \
  "scipy==1.11.4" \
  "scikit-image==0.22.0" \
  "imageio==2.34.2" \
  "tifffile==2024.8.30" \
  "Shapely==2.0.6" \
  "lazy-loader==0.4"

python -m pip install -q --no-deps "imgaug==0.4.0"

echo "Re-pinning NumPy/OpenCV after dependency installs..."
python -m pip install -q --force-reinstall --no-cache-dir \
  "numpy==1.26.4" \
  "opencv-python-headless==4.10.0.84"

In [ ]:
# Clone PARSeq.
import shutil
from pathlib import Path

if PARSEQ_DIR.exists():
    print(f"Removing existing repo at {PARSEQ_DIR}")
    shutil.rmtree(PARSEQ_DIR)

!git clone --depth 1 https://github.com/baudm/parseq.git "{PARSEQ_DIR}"
!python -m pip install -q -e "{PARSEQ_DIR}"

In [ ]:
# Apply the hybrid LaTeX tokenizer patch from Google Drive into the cloned PARSeq repo.
# This replaces the older embedded-patch cell and uses the zip/folder you placed on Drive.

from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

PARSEQ_DIR = Path("/content/parseq")
PATCH_SOURCE = Path(PATCH_SOURCE_STR)
PATCH_EXTRACT_DIR = Path("/content/parseq_hybrid_patch_extract")

# Common locations based on your Drive screenshot and previous patch filename.
patch_candidates = [
    PATCH_SOURCE,
    Path("/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch_20260524.zip"),
    Path("/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch.zip"),
    Path("/content/drive/MyDrive/Parseq/parseq_hybrid_tokenizer_patch"),
    DRIVE_TRAINING_DIR.parent / "parseq_hybrid_tokenizer_patch_20260524.zip",
    DRIVE_TRAINING_DIR.parent / "parseq_hybrid_tokenizer_patch.zip",
    DRIVE_TRAINING_DIR.parent / "parseq_hybrid_tokenizer_patch",
    DRIVE_TRAINING_DIR / "parseq_hybrid_tokenizer_patch_20260524.zip",
    DRIVE_TRAINING_DIR / "parseq_hybrid_tokenizer_patch.zip",
    DRIVE_TRAINING_DIR / "parseq_hybrid_tokenizer_patch",
]

# Remove duplicates while preserving order.
seen = set()
patch_candidates = [p for p in patch_candidates if not (str(p) in seen or seen.add(str(p)))]

source = next((p for p in patch_candidates if p.exists()), None)
if source is None:
    print("Tried these patch paths:")
    for p in patch_candidates:
        print("  -", p)
    raise FileNotFoundError(
        "Could not find parseq_hybrid_tokenizer_patch on Drive. "
        "Set PATCH_SOURCE_STR in the config cell to the zip file or extracted folder."
    )

print("Using patch source:", source)

if source.is_file():
    if source.suffix.lower() != ".zip":
        raise ValueError(f"Patch source is a file but not a .zip: {source}")
    if PATCH_EXTRACT_DIR.exists():
        shutil.rmtree(PATCH_EXTRACT_DIR)
    PATCH_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source, "r") as zf:
        zf.extractall(PATCH_EXTRACT_DIR)
    search_root = PATCH_EXTRACT_DIR
else:
    search_root = source

# Find the actual patch root. The zip may contain a top-level parseq_hybrid_tokenizer_patch/ folder.
def is_patch_root(path: Path) -> bool:
    return (
        (path / "strhub" / "data" / "latex_tokenizer.py").exists()
        and (path / "configs" / "charset" / "latex_hybrid.yaml").exists()
    )

if is_patch_root(search_root):
    patch_root = search_root
else:
    matches = [p for p in search_root.rglob("latex_tokenizer.py") if is_patch_root(p.parents[2])]
    if not matches:
        print("Patch tree preview:")
        for p in list(search_root.rglob("*"))[:80]:
            print("  ", p)
        raise FileNotFoundError(
            "Could not find strhub/data/latex_tokenizer.py plus configs/charset/latex_hybrid.yaml "
            "inside the patch zip/folder."
        )
    patch_root = matches[0].parents[2]

print("Patch root:", patch_root)

# Copy the intended patch files. Missing optional files are skipped.
required_files = [
    "strhub/data/latex_tokenizer.py",
    "strhub/models/base.py",
    "strhub/models/parseq/system.py",
    "strhub/models/parseq/model.py",
    "configs/model/parseq.yaml",
    "configs/charset/latex_hybrid.yaml",
]
optional_files = [
    "train.py",
    "strhub/data/utils.py",
    "configs/experiment/parseq-tiny.yaml",
    "configs/charset/94_full.yaml",
]

copied = []
for rel in required_files + optional_files:
    src = patch_root / rel
    dst = PARSEQ_DIR / rel
    if not src.exists():
        if rel in required_files:
            raise FileNotFoundError(f"Required patch file is missing: {src}")
        print("Skipping optional missing file:", rel)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied.append(rel)
    print("Copied", rel)

# Remove stale bytecode that could reference old code.
for cache_dir in PARSEQ_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

# Compile the changed Python files so syntax errors surface before training starts.
compile_targets = [
    PARSEQ_DIR / "train.py",
    PARSEQ_DIR / "strhub" / "data" / "latex_tokenizer.py",
    PARSEQ_DIR / "strhub" / "models" / "base.py",
    PARSEQ_DIR / "strhub" / "models" / "parseq" / "system.py",
    PARSEQ_DIR / "strhub" / "models" / "parseq" / "model.py",
]
subprocess.run([sys.executable, "-m", "py_compile", *map(str, compile_targets)], check=True)

# Tokenizer smoke test.
sys.path.insert(0, str(PARSEQ_DIR))
from omegaconf import OmegaConf
from strhub.data.latex_tokenizer import HybridLatexTokenizer

cfg = OmegaConf.load(PARSEQ_DIR / "configs" / "charset" / "latex_hybrid.yaml")
tok = HybridLatexTokenizer(str(cfg.model.charset_train), list(cfg.model.latex_tokens))
examples = [r"x_{2}", r"y^{(0 + 0)}", r"\sum_{n=1}^{5}", r"\frac{x+1}{\alpha}"]
print("\nTokenizer smoke test:")
for ex in examples:
    print(f"  {ex!r} -> {tok.tokenize(ex)}")

print("\nCopied files:")
for rel in copied:
    print("  -", rel)

print("\nPatch applied successfully.")


In [ ]:
# Verify installed versions/imports, then choose GPU/CPU training.
import numpy, cv2, torch, imgaug
import pytorch_lightning, hydra, omegaconf, timm, lmdb

print("NumPy:             ", numpy.__version__)
print("OpenCV:            ", cv2.__version__)
print("Torch:             ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:               ", torch.cuda.get_device_name(0))
print("imgaug:            ", imgaug.__version__)
print("pytorch_lightning: ", pytorch_lightning.__version__)
print("hydra:             ", hydra.__version__)
print("omegaconf:         ", omegaconf.__version__)
print("timm:              ", timm.__version__)
print("lmdb OK")

assert numpy.__version__ == "1.26.4", "NumPy version is not pinned to 1.26.4"
assert cv2.__version__ == "4.10.0", "OpenCV version is not pinned to 4.10.0"
assert imgaug.__version__ == "0.4.0", "imgaug version is not 0.4.0"

ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1
print("Training accelerator:", ACCELERATOR)

## Copy the LMDB dataset from Drive to local Colab storage

Training from `/content` is usually faster and less flaky than training directly from Google Drive.

This cell copies:

```text
/content/drive/MyDrive/Parseq/Training/data
```

to:

```text
/content/parseq_data
```

In [ ]:
# Copy dataset from Drive to local runtime storage.
import os, shutil, subprocess
from pathlib import Path

if not DATA_SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"DATA_SOURCE_DIR does not exist: {DATA_SOURCE_DIR}\n"
        "Edit DRIVE_TRAINING_DIR or DATA_SOURCE_DIR in the config cell."
    )

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)

WORK_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

src = str(DATA_SOURCE_DIR) + "/"
dst = str(WORK_DATA_DIR) + "/"

print(f"Copying dataset:\n  {src}\n→ {dst}")

try:
    subprocess.run(["rsync", "-a", "--info=progress2", src, dst], check=True)
except Exception as exc:
    print("rsync failed; falling back to shutil.copytree:", exc)
    shutil.copytree(DATA_SOURCE_DIR, WORK_DATA_DIR)

print("\nLocal dataset folders:")
!find "{WORK_DATA_DIR}" -maxdepth 4 -type f \( -name "data.mdb" -o -name "lock.mdb" \) -print

## Optional: inspect LMDB counts and label problems

This catches the common reasons PARSeq silently trains/evaluates on fewer samples than expected:

- labels longer than `model.max_label_length`
- labels containing characters/tokens outside the selected charset/tokenizer
- missing/incorrect LMDB structure

Because `data.remove_whitespace=false`, spaces are preserved and counted.

For `charset=latex_hybrid`, length is reported as **hybrid-token length**, not raw character length.


In [ ]:
# Inspect LMDB sample counts, label lengths, and charset/tokenizer compatibility.
from pathlib import Path
from omegaconf import OmegaConf
import lmdb
from collections import Counter
import sys

sys.path.insert(0, str(PARSEQ_DIR))

charset_cfg_path = PARSEQ_DIR / "configs" / "charset" / f"{CHARSET}.yaml"
if not charset_cfg_path.exists():
    raise FileNotFoundError(f"Could not find charset config: {charset_cfg_path}")

cfg = OmegaConf.load(charset_cfg_path)
charset_train = str(cfg.model.charset_train)
tokenizer_type = str(cfg.model.get("tokenizer_type", "char"))
latex_tokens = list(cfg.model.get("latex_tokens", []))
allowed_chars = set(charset_train)

if tokenizer_type == "latex_hybrid":
    from strhub.data.latex_tokenizer import HybridLatexTokenizer
    label_tokenizer = HybridLatexTokenizer(charset_train, latex_tokens)
else:
    label_tokenizer = None

print("Charset:", CHARSET)
print("tokenizer_type:", tokenizer_type)
print("charset_train length:", len(charset_train))
print("charset_train repr:", repr(charset_train))
print("latex token count:", len(latex_tokens))
if latex_tokens:
    print("latex tokens:", latex_tokens)
print("MAX_LABEL_LENGTH:", MAX_LABEL_LENGTH)
print("Length mode:", "hybrid tokens" if label_tokenizer else "raw characters")

def read_lmdb_labels(db_file: Path):
    env = lmdb.open(
        str(db_file.parent),
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False,
        max_readers=1,
    )
    labels = []
    with env.begin(write=False) as txn:
        raw_n = txn.get(b"num-samples")
        if raw_n is None:
            # Fallback: scan labels if num-samples is missing.
            with txn.cursor() as cur:
                for key, value in cur:
                    if key.startswith(b"label-"):
                        labels.append(value.decode("utf-8", errors="replace"))
            return labels

        n = int(raw_n.decode())
        for i in range(1, n + 1):
            raw = txn.get(f"label-{i:09d}".encode())
            if raw is None:
                labels.append(None)
            else:
                labels.append(raw.decode("utf-8", errors="replace"))
    env.close()
    return labels

def label_units(label: str):
    if label_tokenizer is not None:
        return label_tokenizer.tokenize(label)
    return list(label)

def inspect_split(split_name: str, root: Path):
    dbs = sorted(root.rglob("data.mdb"))
    print(f"\n=== {split_name} ===")
    if not dbs:
        print("No data.mdb files found under", root)
        return

    total = 0
    missing = 0
    too_long = []
    bad_charset = []

    for db in dbs:
        labels = read_lmdb_labels(db)
        total += len(labels)
        print(f"{db.parent}: {len(labels)} samples")
        for idx, label in enumerate(labels, start=1):
            if label is None:
                missing += 1
                continue
            try:
                units = label_units(label)
            except Exception as exc:
                bad_charset.append((db.parent, idx, [str(exc)], label))
                continue

            if len(units) > MAX_LABEL_LENGTH:
                too_long.append((db.parent, idx, len(units), len(label), label, units))

            if label_tokenizer is None:
                bad = sorted({ch for ch in label if ch not in allowed_chars})
                if bad:
                    bad_charset.append((db.parent, idx, bad, label))

    print(f"Total raw labels: {total}")
    print(f"Missing labels:   {missing}")
    print(f"Too long:         {len(too_long)}")
    print(f"Bad charset/tokenizer: {len(bad_charset)}")

    if too_long:
        print("\nLongest labels:")
        for db_parent, idx, token_n, char_n, label, units in sorted(too_long, key=lambda x: x[2], reverse=True)[:20]:
            print(f"  {db_parent.name} #{idx:>6} | tokens={token_n:>3} | chars={char_n:>3} | {label!r}")
            if label_tokenizer is not None:
                print(f"      units={units}")

    if bad_charset:
        print("\nFirst bad charset/tokenizer examples:")
        for db_parent, idx, bad, label in bad_charset[:20]:
            print(f"  {db_parent.name} #{idx:>6} | bad={bad} | {label!r}")

inspect_split("train", WORK_DATA_DIR / "train")
inspect_split("val", WORK_DATA_DIR / "val")


## Build the training command

This mirrors your Mac command, with these Colab changes:

- `data.root_dir=/content/parseq_data`
- `trainer.accelerator=gpu` when a GPU is available
- output/checkpoints saved under your Drive `runs/` folder
- `charset=latex_hybrid` by default, using the Drive patch
- `model.max_label_length` is hybrid-token length when `charset=latex_hybrid`

The `+pretrained_load=auto` line is kept as a safe extra config flag. Patched `train.py` versions that support it will use it; patched versions that do not support it will ignore the extra field.


In [ ]:
# Create a runnable training script.
from pathlib import Path
import shlex, os, textwrap

RUNS_DIR.mkdir(parents=True, exist_ok=True)

cmd_lines = [
    "HYDRA_FULL_ERROR=1 python train.py",
    "  +experiment=parseq-tiny",
    "  pretrained=parseq-tiny",
    f"  charset={CHARSET}",
    "  'model.charset_test=${model.charset_train}'",
    f"  data.root_dir={shlex.quote(str(WORK_DATA_DIR))}",
    "  data.train_dir=real",
    "  data.remove_whitespace=false",
    "  data.normalize_unicode=false",
    "  data.augment=true",
    f"  data.num_workers={NUM_WORKERS}",
    f"  model.batch_size={BATCH_SIZE}",
    f"  model.max_label_length={MAX_LABEL_LENGTH}",
    f"  trainer.accelerator={ACCELERATOR}",
    f"  trainer.devices={DEVICES}",
    f"  trainer.max_epochs={MAX_EPOCHS}",
    "  trainer.val_check_interval=1.0",
    f"  '+trainer.default_root_dir={str(RUNS_DIR)}'",
    f"  +trainer.log_every_n_steps={LOG_EVERY_N_STEPS}",
    f"  'hydra.run.dir={str(RUNS_DIR)}/${{now:%Y-%m-%d_%H-%M-%S}}'",
    f"  +pretrained_load={PRETRAINED_LOAD}",
    "  '+trainer.enable_progress_bar=false'",
]

script = "#!/usr/bin/env bash\nset -euo pipefail\ncd " + shlex.quote(str(PARSEQ_DIR)) + "\n\n"
script += " \\\n".join(cmd_lines) + "\n"

script_path = Path("/content/train_parseq_colab.sh")
script_path.write_text(script)
script_path.chmod(0o755)

print(script)
print(f"\nWrote: {script_path}")


In [ ]:
# Run training.
!bash /content/train_parseq_colab.sh

In [ ]:
# save/copy/download anything important first!

from google.colab import runtime
runtime.unassign()

## Checkpoints and metrics

Run these after training finishes, or after interrupting a long run.

In [ ]:
# List checkpoints saved to Drive.
!find "{RUNS_DIR}" -name "*.ckpt" -print | sort

In [ ]:
# Print the final scalar metrics from the most recent TensorBoard event file.
from pathlib import Path
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = sorted(
    RUNS_DIR.rglob("events.out.tfevents*"),
    key=lambda p: p.stat().st_mtime
)

if not event_files:
    raise SystemExit(f"No TensorBoard event files found under {RUNS_DIR}")

event_file = event_files[-1]
print(f"Reading TensorBoard event file:\n  {event_file}\n")

ea = EventAccumulator(str(event_file))
ea.Reload()

scalar_tags = ea.Tags().get("scalars", [])
if not scalar_tags:
    raise SystemExit("No scalar metrics found in the TensorBoard event file.")

print("Final logged metrics:")
print("-" * 72)
for tag in sorted(scalar_tags):
    values = ea.Scalars(tag)
    if not values:
        continue
    last = values[-1]
    print(f"{tag:40s} {last.value:.6f}    step={last.step}")
print("-" * 72)

In [ ]:
# TensorBoard viewer.
# Run this cell, then refresh/rerun it after new logs appear.
import os
os.environ["TB_LOGDIR"] = str(RUNS_DIR)
%load_ext tensorboard
%tensorboard --logdir "$TB_LOGDIR"

## Optional: resume from a checkpoint

Paste a `.ckpt` path from the checkpoint-listing cell into `CKPT_PATH`, then run the next cell.

Example:

```python
CKPT_PATH = "/content/drive/MyDrive/Parseq/Training/runs/2026-05-23_12-34-56/checkpoints/last.ckpt"
```

In [ ]:
#@title Optional resume command builder
from pathlib import Path
import shlex

CKPT_PATH = ""  #@param {type:"string"}

if not CKPT_PATH:
    raise SystemExit("Set CKPT_PATH first.")

ckpt = Path(CKPT_PATH)
if not ckpt.exists():
    raise FileNotFoundError(ckpt)

resume_lines = [
    "HYDRA_FULL_ERROR=1 python train.py",
    "  +experiment=parseq-tiny",
    "  pretrained=parseq-tiny",
    f"  charset={CHARSET}",
    "  'model.charset_test=${model.charset_train}'",
    f"  data.root_dir={shlex.quote(str(WORK_DATA_DIR))}",
    "  data.train_dir=real",
    "  data.remove_whitespace=false",
    "  data.normalize_unicode=false",
    "  data.augment=true",
    f"  data.num_workers={NUM_WORKERS}",
    f"  model.batch_size={BATCH_SIZE}",
    f"  model.max_label_length={MAX_LABEL_LENGTH}",
    f"  trainer.accelerator={ACCELERATOR}",
    f"  trainer.devices={DEVICES}",
    f"  trainer.max_epochs={MAX_EPOCHS}",
    "  trainer.val_check_interval=1.0",
    f"  '+trainer.default_root_dir={str(RUNS_DIR)}'",
    f"  +trainer.log_every_n_steps={LOG_EVERY_N_STEPS}",
    f"  'hydra.run.dir={str(RUNS_DIR)}/${{now:%Y-%m-%d_%H-%M-%S}}'",
    f"  +pretrained_load={PRETRAINED_LOAD}",
    f"  ckpt_path={shlex.quote(str(ckpt))}",
]

resume_script = "#!/usr/bin/env bash\nset -euo pipefail\ncd " + shlex.quote(str(PARSEQ_DIR)) + "\n\n"
resume_script += " \\\n".join(resume_lines) + "\n"

resume_script_path = Path("/content/resume_parseq_colab.sh")
resume_script_path.write_text(resume_script)
resume_script_path.chmod(0o755)

print(resume_script)
print(f"\nWrote: {resume_script_path}")


In [ ]:
# Run this only after building the resume script above.
!bash /content/resume_parseq_colab.sh

## Notes

- If the install/import cell gives a NumPy/OpenCV/imgaug import error, use `Runtime → Restart runtime`, then rerun the notebook from the configuration cell downward.
- If the Drive patch cell cannot find the patch, set `PATCH_SOURCE_STR` to either the zip path or the extracted folder path.
- If the LMDB inspection shows fewer usable validation samples than expected, look first at **Too long** and **Bad charset/tokenizer**.
- With `charset=latex_hybrid`, label length is counted in hybrid tokens. For example, `\frac{x+1}{\alpha}` is shorter in tokens than in raw characters.
- Your uploaded patch `train.py` may still enable SWA. If you want SWA disabled, edit `train.py` after the patch cell and change the callback list to `callbacks=[checkpoint]`.
